# Experimento 04 — HistGradientBoosting

Modelo boosting compatible con el entorno. One-hot denso evita la incompatibilidad del estimador con matrices sparse.

Sólo se utilizan train y validation. Test no se carga.

In [1]:
from pathlib import Path
import json,time
from datetime import datetime,timezone
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score,balanced_accuracy_score,classification_report,confusion_matrix,f1_score,precision_score,recall_score
ROOT=Path.cwd().parent if Path.cwd().name=="experimentos" else Path.cwd()
DATA=ROOT/"data/processed/modelado/dataset_modelado_nivel_actividad_firms.parquet"
SPLITS=ROOT/"data/processed/modelado/splits";OUT=ROOT/"results/experimentos_parciales";OUT.mkdir(parents=True,exist_ok=True)
datos=pd.read_parquet(DATA);datos.fecha_inicio_semana=pd.to_datetime(datos.fecha_inicio_semana)
KEY=["departamento","fecha_inicio_semana"];TARGET="nivel_actividad_firms";ORDER=["Sin actividad","Bajo","Moderado","Alto"];RS=42
def subset(name):
 k=pd.read_parquet(SPLITS/f"{name}_keys.parquet");k.fecha_inicio_semana=pd.to_datetime(k.fecha_inicio_semana)
 return datos.merge(k,on=KEY,how="inner",validate="one_to_one")
train=subset("train");validation=subset("validation")
assert len(train)==5546 and len(validation)==1188
CAT=["departamento","mes"]
MET=["temperature_2m_max_mean_weekly","relative_humidity_2m_min_mean_weekly","wind_speed_10m_max_mean_weekly","precipitation_sum_weekly"]
LAGS=[f"cantidad_detecciones_lag{i}" for i in range(1,5)]
ROLL=["promedio_detecciones_ultimas_4_semanas","max_detecciones_ultimas_4_semanas","semanas_con_actividad_ultimas_4"]
VARIANTS={"V1":CAT+MET,"V2":CAT+MET+LAGS,"V3":CAT+MET+ROLL,"V4":CAT+MET+LAGS+ROLL}
PROHIBITED={"cantidad_detecciones","fecha_inicio_semana","anio","numero_semana",TARGET,"nivel_actividad_firms_codigo"}
assert all(PROHIBITED.isdisjoint(v) for v in VARIANTS.values())
def metrics(y,p):
 r=classification_report(y,p,labels=ORDER,output_dict=True,zero_division=0)
 return {"accuracy_validation":accuracy_score(y,p),"balanced_accuracy_validation":balanced_accuracy_score(y,p),"precision_macro_validation":precision_score(y,p,average="macro",zero_division=0),"recall_macro_validation":recall_score(y,p,average="macro",zero_division=0),"f1_macro_validation":f1_score(y,p,average="macro",zero_division=0),"f1_weighted_validation":f1_score(y,p,average="weighted",zero_division=0),**{f"precision_{x.lower().replace(' ','_')}":r[x]["precision"] for x in ORDER},**{f"recall_{x.lower().replace(' ','_')}":r[x]["recall"] for x in ORDER},**{f"f1_{x.lower().replace(' ','_')}":r[x]["f1-score"] for x in ORDER}}
def row(eid,model,variant,params,pred,seconds,note):return {"experiment_id":eid,"modelo":model,"variante_variables":variant,"hiperparametros":json.dumps(params,sort_keys=True,ensure_ascii=False),**metrics(validation[TARGET],pred),"duracion_segundos":seconds,"fecha_ejecucion":datetime.now(timezone.utc).isoformat(),"notebook_origen":NOTEBOOK,"observaciones":note}

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
NOTEBOOK="Experimento_04_HistGradientBoosting.ipynb"
def build(cols,**params):
 cat=[x for x in cols if x in CAT];num=[x for x in cols if x not in CAT]
 pre=ColumnTransformer([("cat",OneHotEncoder(handle_unknown="ignore",sparse_output=False),cat),("num","passthrough",num)])
 defaults=dict(random_state=RS);defaults.update(params)
 return Pipeline([("pre",pre),("model",HistGradientBoostingClassifier(**defaults))])

In [2]:
rows=[];base_predictions={}
for v,cols in VARIANTS.items():
 t=time.perf_counter();mod=build(cols).fit(train[cols],train[TARGET]);pred=mod.predict(validation[cols]);elapsed=time.perf_counter()-t;base_predictions[v]=pred
 rows.append(row(f"hgb_base_{v.lower()}","HistGradientBoosting",v,{"random_state":RS},pred,elapsed,"Configuración base sklearn"))
base=pd.DataFrame(rows);base[["variante_variables","f1_macro_validation","balanced_accuracy_validation","f1_bajo","f1_moderado","f1_alto"]].round(4)
for v,pred_v in base_predictions.items():
 print(f"Matriz de confusión validation - {v}")
 display(pd.DataFrame(confusion_matrix(validation[TARGET],pred_v,labels=ORDER),index=ORDER,columns=ORDER))

Matriz de confusión validation - V1


,Sin actividad,Bajo,Moderado,Alto
Sin actividad,678,28,33,6
Bajo,146,13,23,3
Moderado,130,14,37,9
Alto,36,3,16,13


Matriz de confusión validation - V2


,Sin actividad,Bajo,Moderado,Alto
Sin actividad,689,15,35,6
Bajo,148,8,27,2
Moderado,134,12,37,7
Alto,33,2,19,14


Matriz de confusión validation - V3


,Sin actividad,Bajo,Moderado,Alto
Sin actividad,680,27,31,7
Bajo,139,16,22,8
Moderado,135,16,29,10
Alto,31,6,14,17


Matriz de confusión validation - V4


,Sin actividad,Bajo,Moderado,Alto
Sin actividad,682,22,34,7
Bajo,142,11,25,7
Moderado,133,16,35,6
Alto,33,5,15,15


In [3]:
best_variant=base.sort_values(["f1_macro_validation","balanced_accuracy_validation"],ascending=False).iloc[0].variante_variables;cols=VARIANTS[best_variant]
search=RandomizedSearchCV(build(cols),{"model__learning_rate":[0.03,0.06,0.1],"model__max_iter":[100,200,300],"model__max_leaf_nodes":[15,31,63],"model__max_depth":[None,5,10],"model__min_samples_leaf":[10,20,40],"model__l2_regularization":[0.0,0.1,1.0]},n_iter=12,scoring="f1_macro",cv=3,random_state=RS,n_jobs=-1,refit=True)
t=time.perf_counter();search.fit(train[cols],train[TARGET]);elapsed=time.perf_counter()-t;pred=search.predict(validation[cols]);best_params={k.replace("model__",""):v for k,v in search.best_params_.items()};best_params["random_state"]=RS
tuned=pd.DataFrame([row(f"hgb_tuned_{best_variant.lower()}","HistGradientBoosting",best_variant,best_params,pred,elapsed,f"RandomizedSearchCV 12 iteraciones, cv=3; best_cv={search.best_score_:.4f}")]);results=pd.concat([base,tuned],ignore_index=True);results.sort_values("f1_macro_validation",ascending=False).round(4)

,experiment_id,modelo,variante_variables,hiperparametros,accuracy_validation,balanced_accuracy_validation,precision_macro_validation,recall_macro_validation,f1_macro_validation,f1_weighted_validation,...,recall_moderado,recall_alto,f1_sin_actividad,f1_bajo,f1_moderado,f1_alto,duracion_segundos,fecha_ejecucion,notebook_origen,observaciones
2,hgb_base_v3,HistGradientBoosting,V3,"{""random_state"": 42}",0.6246,0.3505,0.4108,0.3505,0.3565,0.5630,...,0.1526,0.2500,0.7861,0.1280,0.2028,0.3091,1.4288,2026-09-30T19:16:39.144526+00:00,Experimento_04_HistGradientBoosting.ipynb,Configuración base sklearn
3,hgb_base_v4,HistGradientBoosting,V4,"{""random_state"": 42}",0.6254,0.3449,0.4106,0.3449,0.3509,0.5615,...,0.1842,0.2206,0.7862,0.0921,0.2341,0.2913,1.5667,2026-09-30T19:16:40.784779+00:00,Experimento_04_HistGradientBoosting.ipynb,Configuración base sklearn
0,hgb_base_v1,HistGradientBoosting,V1,"{""random_state"": 42}",0.6237,0.3416,0.4169,0.3416,0.3497,0.5614,...,0.1947,0.1912,0.7816,0.1070,0.2475,0.2626,1.6360,2026-09-30T19:16:36.009706+00:00,Experimento_04_HistGradientBoosting.ipynb,Configuración base sklearn
1,hgb_base_v2,HistGradientBoosting,V2,"{""random_state"": 42}",0.6296,0.3422,0.4247,0.3422,0.3472,0.5603,...,0.1947,0.2059,0.7879,0.0721,0.2403,0.2887,1.5781,2026-09-30T19:16:37.652182+00:00,Experimento_04_HistGradientBoosting.ipynb,Configuración base sklearn
4,hgb_tuned_v3,HistGradientBoosting,V3,"{""l2_regularization"": 0.0, ""learning_rate"": 0....",0.6229,0.3380,0.4024,0.3380,0.3427,0.5563,...,0.1526,0.2206,0.7869,0.0913,0.1986,0.2941,33.2306,2026-09-30T19:17:14.173188+00:00,Experimento_04_HistGradientBoosting.ipynb,"RandomizedSearchCV 12 iteraciones, cv=3; best_..."


In [4]:
display(pd.DataFrame(confusion_matrix(validation[TARGET],pred,labels=ORDER),index=ORDER,columns=ORDER));display(pd.DataFrame(classification_report(validation[TARGET],pred,labels=ORDER,output_dict=True,zero_division=0)).T.loc[ORDER].round(4));results.to_csv(OUT/"04_hist_gradient_boosting.csv",index=False)

,Sin actividad,Bajo,Moderado,Alto
Sin actividad,685,25,30,5
Bajo,139,11,30,5
Moderado,137,15,29,9
Alto,35,5,13,15


,precision,recall,f1-score,support
Sin actividad,0.6878,0.9195,0.7869,745.0
Bajo,0.1964,0.0595,0.0913,185.0
Moderado,0.2843,0.1526,0.1986,190.0
Alto,0.4412,0.2206,0.2941,68.0
